<div style="border-top:4px solid #0f766e;padding:28px 0 18px">
<div style="color:#0f766e;font-size:13px;font-weight:700">LAB 01 · DATA WAREHOUSING WITH APACHE DORIS</div>
<h1>Connect to Doris and Query WWI Historical Orders</h1>
<p>Start learning Doris with ten historical orders you can verify, then bulk-load the complete business tables in Module 5.</p>
<p>About 25 minutes · Transformed subset of the official WWI sample · Target Doris 4.1.3</p></div>

[Course notes](course1_introduction_to_apache_doris.md) · [Quiz](quiz1_doris_fundamentals.ipynb) · [Course contents](../README.md)

This lab uses simulated wholesale business data from Microsoft Wide World Importers. You will insert ten orders into Doris, query order details, and summarize pre-tax order amounts by date.

The sample files are downloaded from the course object-storage bucket under the MIT license. See the [data notes](../../datasets/README.md).


### Initialize the Lab Tools

Run the next cell to load the Python tools and display styles. When "Lab tools loaded" appears, continue to Section 1 to start and connect to Doris. After restarting the kernel, rerun from here.

If ModuleNotFoundError appears, install the dependencies following the [environment instructions](../../environments/single-node/README.md) and confirm that the Notebook uses the corresponding Python kernel.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.runtime import expect, fixture
from dw_course.wwi import HISTORY_COLUMNS, history_rows, sample
from dw_course.docker_runtime import prepare_environment, connect_sandbox
from dw_course.ui import card, install_styles

install_styles()
card("Continue to Section 1 and run the startup code to start and connect to Doris.", "ok", "Lab tools loaded")


## 1. Start the Single-Container Lab Environment

This course uses the official All-in-One image to run one FE and one BE in a single container.
This is a storage-compute integrated sandbox for learning, not a production deployment. Later labs continue to use the same container and lab database.

Before starting, launch Docker Desktop (macOS) or Docker Engine (Linux), and install the Compose plugin.
Reserve 4 CPUs, 8 GB of memory, and 20 GB of free disk space; the initial image download and startup may take several minutes.

**Running the following startup code creates or starts the course sandbox and connects to the lab database.**
The default lab database is dw_course_l1_demo. Step 4 drops and recreates **orders_sample** in it;
other tables are outside this section's reset scope. First confirm that this teaching sample table can be reset.

The course tools configure the connection address automatically; you do not need to enter FE_HOST or FE_PORT, or choose a connection method.
**Docker and Doris run on the machine hosting the Jupyter kernel, which is not necessarily the Mac running your browser.**


### Start and Connect

The next cell validates the Compose configuration, starts the container, waits for health checks, and verifies FE connectivity and BE computation, in that order.
An existing course container continues using its named volumes; no metadata or table data is deleted, and no other services are stopped.

After connecting, the tools create and select the lab database, set the session time zone to +08:00, and disable Group Commit for this session
so you can observe each write result. Later labs only connect to this sandbox; they do not each start another container.


In [ ]:
prepare_environment(start=True)
lab = connect_sandbox()
lab.sql("SELECT 1 AS connection_ok, DATABASE() AS current_database", title="Connection and current database");


**Expected result:** connection_ok is 1, and the default current_database is dw_course_l1_demo.

**Troubleshooting:**

- Docker startup fails: Confirm that Docker is running and the Compose plugin is available, then check the container logs.
- Port in use: Ask the instructor to coordinate the environment; do not stop services that do not belong to this course.
- Connection fails: Wait for the container health check to pass, then rerun this section; do not switch to another database.

For commands and ports, see the [environment instructions](../../environments/single-node/README.md).


## 2. Check FE and BE

FE receives SQL and plans queries; BE executes queries and also stores internal table data in this section's storage-compute integrated environment.
Confirm that the nodes are alive before writing data.


In [ ]:
lab.sql("SELECT VERSION() AS protocol_version, @@version_comment AS version_comment", title="Connection version information");
lab.sql("SHOW FRONTENDS", title="FE nodes");
lab.sql("SHOW BACKENDS", title="BE nodes");


**Expected result:** The course sandbox has one live FE and one live BE, with Alive set to true.
Record the FE/BE Version fields and check them against the course image version; SELECT VERSION() may return a protocol compatibility version.

If BE is not alive, resolve the environment issue first. Do not keep creating tables and inserting data to see whether it happens to work.


## 3. Understand the WWI Order Sample

Select five orders each from 2013-01-01 and 2013-01-02 in WWI Orders, and aggregate OrderLines by order to create an introductory sample with one row per order. Module 5 loads the original multi-table data.

| Field | Meaning |
| --- | --- |
| order_id, customer_id | Original WWI business identifiers |
| order_date | Source order date |
| order_amount | Sum of Quantity × UnitPrice across detail lines, representing the pre-tax order amount |
| line_count | Number of item detail lines for the order |
| data_source | Data source, WWI for this batch |

The ten orders have a pre-tax total of 12220.60. Payment status requires separate payment or account data.
The ten-order WWI sample is downloaded from the course object-storage bucket on first use; order IDs are 1–5 and 80–84.


## 4. Create Your First Internal Table

The complete DDL is shown below: DECIMAL stores amounts, and DATE stores original order dates.
DUPLICATE KEY(order_id) is used for sorting and does not enforce uniqueness; one bucket and one replica are used for this section's small sample.

**Reset notice:** The next cell drops only orders_sample in the current lab database, not the entire database.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_sample")
lab.execute("""
CREATE TABLE orders_sample (
    order_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    order_date DATE NOT NULL,
    order_amount DECIMAL(18,2) NOT NULL,
    line_count INT NOT NULL,
    data_source VARCHAR(32) NOT NULL
)
DUPLICATE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.sql("DESC orders_sample", title="Historical order projection fields");
lab.sql("SHOW CREATE TABLE orders_sample", title="Actual table definition");


**Expected result:** Six fields, Duplicate Key, one bucket, and one replica. The table is created, but no data has been inserted yet.


## 5. Insert Ten Historical Orders

An explicit INSERT makes the fields clear; Module 5 loads all business tables from files. The data comes from WWI, with amounts aggregated from the original detail lines.
Do not rerun this cell on its own: Duplicate Key appends data. To rerun the whole process, execute Step 4 first.


In [ ]:
lab.execute("""
INSERT INTO orders_sample (
    order_id, customer_id, order_date, order_amount, line_count, data_source
) VALUES
(1, 832, '2013-01-01', 2300.00, 1, 'WWI'),
(2, 803, '2013-01-01', 405.00, 2, 'WWI'),
(3, 105, '2013-01-01', 90.00, 1, 'WWI'),
(4, 57, '2013-01-01', 445.20, 3, 'WWI'),
(5, 905, '2013-01-01', 704.00, 3, 'WWI'),
(80, 543, '2013-01-02', 1138.00, 3, 'WWI'),
(81, 456, '2013-01-02', 376.00, 2, 'WWI'),
(82, 487, '2013-01-02', 234.00, 2, 'WWI'),
(83, 154, '2013-01-02', 6220.40, 4, 'WWI'),
(84, 111, '2013-01-02', 308.00, 2, 'WWI')
""")
lab.sql("SELECT * FROM orders_sample ORDER BY order_id", title="Check the ten WWI orders");


**Expected result:** Order IDs 1–5 and 80–84, ten rows in total; order 1 has a pre-tax amount of 2300.00. If twenty rows appear, check whether INSERT was repeated, then rebuild within the stated scope.


## 6. Check Both Totals and Details

COUNT(*) is the projection row count; it equals the order count only when there is one row per order. Counting OrderLines directly gives the number of item detail lines.
Compare every field against the fixed sample in the repository so that coincidentally matching totals do not hide errors.


In [ ]:
lab.sql("SELECT COUNT(*) AS order_count, SUM(order_amount) AS order_amount FROM orders_sample", title="Pre-tax order amount")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_sample"), [(10, "12220.60")])
expect(lab.query("""
SELECT order_id, customer_id, CAST(order_date AS STRING), order_amount, line_count, data_source
FROM orders_sample ORDER BY order_id
"""), history_rows())


**Expected result:** Ten rows, 12220.60, with all fields matching. This result represents only the pre-tax order amount, not payments received, tax-inclusive invoice amounts, or profit.


## 7. Answer the Business Question: Daily Orders

Group and sort by order date to observe how the ten sample orders' counts and amounts are distributed across the two days. The scope is limited to this section's ten orders.


In [ ]:
daily_sql = """
SELECT CAST(order_date AS STRING) AS order_date, COUNT(*) AS order_count,
       SUM(order_amount) AS order_amount
FROM orders_sample GROUP BY order_date ORDER BY order_date
"""
lab.sql(daily_sql, title="Daily summary of sample orders")
expect(lab.query(daily_sql), [("2013-01-01", 5, "3944.20"), ("2013-01-02", 5, "8276.40")])


**Expected result:** 2013-01-01: five orders, 3944.20; 2013-01-02: five orders, 8276.40.


## 8. Your turn: Find Orders with a Pre-tax Amount of at Least 1000.00

First write SQL to return order_id, order_date, and order_amount, then sort by order ID.
Expect three orders totaling 9658.40. A reference answer follows; try it yourself first.


In [ ]:
lab.sql("""
SELECT order_id, order_date, order_amount FROM orders_sample
WHERE order_amount >= 1000.00 ORDER BY order_id
""", title="Amount filter: Reference answer")
expect(lab.query("SELECT order_id FROM orders_sample WHERE order_amount >= 1000.00 ORDER BY order_id"),
       [(1,), (80,), (83,)])
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_sample WHERE order_amount >= 1000.00"),
       [(3, "9658.40")])


## Independent exercise

Count the orders on 2013-01-02 with an amount of at least 300 and total their pre-tax amounts. Decide which conditions belong in WHERE before writing the query. Expect 4 orders and 8042.40.

Write and run your code in the next cell, then expand the reference solution after finishing. A blank exercise is not automatically marked as complete.


In [ ]:
# Write your SQL or load request here.


<details>
<summary>Reference solution (expand after completing the exercise)</summary>

```python
query = """SELECT COUNT(*) AS orders, SUM(order_amount) AS amount
FROM orders_sample
WHERE order_date = '2013-01-02' AND order_amount >= 300"""
lab.sql(query, title="Orders on the second day with an amount of at least 300")
expect(lab.query(query), [(4, "8042.40")], title="Date and amount filters are correct")
```

</details>


## Lab Complete

You have connected to Doris, checked the nodes, created an order table, and checked the details and daily summaries of ten historical orders.

Continue to [Quiz 1](quiz1_doris_fundamentals.ipynb), then move on to [Module 2](../module02-architecture/course2_doris_architecture.md).
For a complete rerun, start from initialization; this section rebuilds only orders_sample. Shutting down the kernel releases the connection, while course data remains in the lab database.
